In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_rows', 25)
%matplotlib inline

from src.data_loading import load_raw_tables, build_transactions
from src.cleaning import clean_transactions
from src.categorization import categorize
from src.config import DATA_DIR
from src.eda import (
    top_products, plot_top_products,
    pareto_analysis, plot_pareto,
    revenue_by_category, plot_category_breakdown,
    revenue_by_month, plot_monthly_revenue,
    revenue_by_weekday, plot_weekday,
    revenue_by_hour, plot_hourly,
    category_seasonality, plot_category_seasonality,
    revenue_by_store, plot_store_revenue,
    store_category_mix, plot_store_category_mix,
)

tables = load_raw_tables(DATA_DIR)
df = categorize(clean_transactions(build_transactions(tables)))
print(f'Ready: {len(df):,} rows  |  {df["name"].nunique()} unique products  |  {df["name_category"].nunique()} categories')

## Block A — Sales EDA
### 1. Top Products by Revenue

In [ ]:
tbl_rev = top_products(df, by='revenue', n=20)
tbl_rev

In [ ]:
fig = plot_top_products(df, by='revenue', n=15)
fig

### 2. Top Products by Units Sold

In [ ]:
tbl_units = top_products(df, by='units', n=15)
tbl_units

In [ ]:
fig = plot_top_products(df, by='units', n=15)
fig

### 3. Pareto Analysis (80 / 20 Rule)

In [ ]:
prod, n80 = pareto_analysis(df)
total_rev = df['revenue'].sum()
top20_share = prod.head(20)['revenue'].sum() / total_rev * 100
print(f'Products to reach 80% of revenue: {n80}')
print(f'Top-20 products share:            {top20_share:.1f}%  (brief anchor: ~80%)')
prod.head(25)

In [ ]:
fig, n80 = plot_pareto(df)
fig

### 4. Revenue by Category

In [ ]:
cat_tbl = revenue_by_category(df)
cat_tbl

In [ ]:
fig = plot_category_breakdown(df)
fig

### 5. Monthly Revenue (Seasonality)

In [ ]:
month_tbl = revenue_by_month(df)
peak = month_tbl.loc[month_tbl['revenue'].idxmax()]
print(f'Peak month: {peak["month_label"]}  —  {peak["revenue"]:,.0f} DKK  ({peak["order_count"]:,} orders)')
month_tbl

In [ ]:
fig = plot_monthly_revenue(df)
fig

### 6. Revenue by Day of Week (local time)

In [ ]:
wd_tbl = revenue_by_weekday(df)
print('Busiest days (by revenue):')
for _, r in wd_tbl.sort_values('revenue', ascending=False).iterrows():
    print(f'  {r["weekday_name"]:<12}  {r["revenue"]:>12,.0f} DKK   {r["order_count"]:>6,} orders   AOV {r["avg_order_value"]:,.0f} DKK')
wd_tbl

In [ ]:
fig = plot_weekday(df)
fig

### 7. Revenue by Hour of Day (local time)

In [ ]:
hr_tbl = revenue_by_hour(df)
top3_hours = hr_tbl.nlargest(3, 'revenue')
print('Top 3 revenue hours (local):')
for _, r in top3_hours.iterrows():
    print(f'  {r["hour"]:02d}:00   {r["revenue"]:>12,.0f} DKK   {r["order_count"]:>5,} orders')
hr_tbl

In [ ]:
fig = plot_hourly(df)
fig

### 8. Category Seasonality Heatmap

In [ ]:
fig = plot_category_seasonality(df)
fig

### 9. Store Performance

In [ ]:
store_tbl = revenue_by_store(df)
store_tbl[['store_short','revenue','order_count','avg_order_value']]

In [ ]:
fig = plot_store_revenue(df)
fig

### 10. Store × Category Mix

In [ ]:
mix = store_category_mix(df)
mix.round(1)

In [ ]:
fig = plot_store_category_mix(df)
fig

### Validation Anchors

In [ ]:
# Top product
top1 = tbl_rev.iloc[0]
assert top1['name'] == '2 kugler'
print(f'Top product:  {top1["name"]}  —  {top1["revenue"]:,.0f} DKK ({top1["pct_revenue"]:.1f}%)  [anchor: ~5.3M, ~20%]')

# Seasonality: peak in May or June
peak_month = month_tbl.loc[month_tbl['revenue'].idxmax(), 'month_label']
assert any(m in peak_month for m in ('May', 'Jun', 'Aug')), f'Unexpected peak: {peak_month}'
print(f'Peak month:   {peak_month}  [anchor: May-Jun]')

# Busiest weekdays: Friday or Saturday
top2_days = wd_tbl.nlargest(2, 'revenue')['weekday_name'].tolist()
assert any(d in top2_days for d in ('Friday', 'Saturday')), f'Top-2 days: {top2_days}'
print(f'Top-2 days:   {top2_days}  [anchor: Fri/Sat]')

# Busiest hours: 12-15 local
top3_hrs = hr_tbl.nlargest(3, 'revenue')['hour'].tolist()
assert any(h in top3_hrs for h in range(11, 16)), f'Top-3 hours: {top3_hrs}'
print(f'Top-3 hours:  {[f"{h}:00" for h in sorted(top3_hrs)]}  [anchor: 12-15]')

print()
print('All validation anchors passed.')